<a href="https://colab.research.google.com/github/corrielynnyuill-debug/AIProject_Stocks-CLY/blob/main/AIProject_Stocks_Part3_CLY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Mount drive in Colab
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the datasets
# Set file path
DATA_PATH = "/content/drive/MyDrive/AIProject/clean_data/"

# Load to DF
df = pd.read_parquet(DATA_PATH + "cleaned_stock_data.parquet")
print(df.head())
print('-'*80)
print(df.columns)


Mounted at /content/drive
  ticker      open     close  adj_close       low      high    volume  \
0      A -0.015269 -0.016592  -0.003238 -0.016609 -0.014101  4.657089   
1      A -0.015912 -0.017590  -0.003238 -0.016657 -0.015771  1.063811   
2      A -0.016320 -0.016592  -0.003238 -0.016592 -0.015532  0.406381   
3      A -0.016022 -0.017693  -0.003238 -0.016609 -0.015622  0.360644   
4      A -0.016618 -0.017401  -0.003238 -0.016609 -0.016024  0.274641   

        date   z_close   z_volume  close_outlier  volume_outlier  rolling_7  \
0 1999-11-18  0.053156  23.254405          False            True        NaN   
1 1999-11-19 -0.120629   4.522274          False            True        NaN   
2 1999-11-22  0.053156   1.095020          False           False        NaN   
3 1999-11-23 -0.138606   0.856593          False           False        NaN   
4 1999-11-24 -0.087669   0.408247          False           False        NaN   

   rolling_30  volatility  RSI         sector  \
0         N

In [2]:
# Feature engineering and technical indicators
# Apply EMA (Exponential Moving Average)
def ema(series, span):
  return series.ewm(span=span, adjust=False).mean()

# MACD (Moving Average Convergence Divergence) implementation
df = df.sort_values(['ticker', 'date'])

# MACD components
df['ema12'] = df.groupby('ticker')['close'].transform(lambda x: ema(x, span=12))
df['ema26'] = df.groupby('ticker')['close'].transform(lambda x: ema(x, span=26))

df['macd'] = df['ema12'] - df['ema26']
df['macd_signal'] = df.groupby('ticker')['macd'].transform(lambda x: ema(x, span=9))
df['macd_hist'] = df['macd'] - df['macd_signal']

# MACD crossovers
df['macd_prev'] = df.groupby('ticker')['macd'].shift(1)
df['macd_signal_prev'] = df.groupby('ticker')['macd_signal'].shift(1)

# Buy when MACD crosses above signal
df['macd_buy'] = (
    (df['macd_prev'] < df['macd_signal_prev']) &
    (df['macd'] > df['macd_signal'])
)

# Sell when MACD crosses below signal
df['macd_sell'] = (
    (df['macd_prev'] > df['macd_signal_prev']) &
    (df['macd'] < df['macd_signal'])
)

# RSI (Relative Strength Index) implementation with EMA
window = 14

# Price changes per ticker
delta = df.groupby('ticker')['close'].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

# EMA of gains and losses
avg_gain = gain.groupby(df['ticker']).transform(
    lambda x: x.ewm(alpha=1/window, adjust=False).mean())
avg_loss = loss.groupby(df['ticker']).transform(
    lambda x: x.ewm(alpha=1/window, adjust=False).mean())

RS = avg_gain / avg_loss
df['RSI'] = 100 - (100 / (1 + RS))

# RSI buy/sell signals
df['RSI_buy'] = df['RSI'] < 30
df['RSI_sell'] = df['RSI'] > 70

# Combine MACD and RSI
df['signal'] = 0 # default hold

df.loc[df['macd_buy'] & df['RSI_buy'], 'signal'] = 1 # Buy
df.loc[df['macd_sell'] & df['RSI_sell'], 'signal'] = -1 # Sell

print(df['signal'].value_counts())


signal
 0    20951021
-1       12693
 1       10175
Name: count, dtype: int64


In [13]:
# Prepare dataset for ML

# reduce dataset to one ticker to avoid RAM crash
ticker = df['ticker'].unique()[0]
df_small = df[df['ticker'] == ticker].copy()

features = [
    'close', 'volume',
    'rolling_7', 'rolling_30', 'volatility',
    'macd', 'macd_signal', 'macd_hist',
    'RSI'
]

df_small = df_small.dropna(subset=features + ['signal']).copy()

# Separate BUY/SELL from HOLD
df_buy_sell = df_small[df_small['signal'] != 0]
df_hold = df_small[df_small['signal'] == 0]

# Sample HOLD rows to match BUY/SELL count (or 2x for stability)
df_hold_sampled = df_hold.sample(
    n=len(df_buy_sell) * 2,
    random_state=42,
    replace=False
)

# Combine into a balanced dataset
df_balanced = pd.concat([df_buy_sell, df_hold_sampled]).sort_values('date')

# Create X and y
X = df_balanced[features]
y = df_balanced['signal']

# Train test split
split_idx = int(len(df_balanced) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

print('-'*80)



--------------------------------------------------------------------------------


In [14]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(
    multi_class='multinomial',
    max_iter=500,
    n_jobs=1
)

log_reg.fit(X_train, y_train)

print('-'*80)


--------------------------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [15]:
# Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    n_jobs=1,
    random_state=42
)

rf_clf.fit(X_train, y_train)

print('-'*80)


--------------------------------------------------------------------------------


In [16]:
# Support Vector Machine (SVM)
from sklearn.svm import SVC

svm = SVC(
    kernel='linear',
    probability=True
)

svm.fit(X_train, y_train)

print('-'*80)


--------------------------------------------------------------------------------


In [17]:
# Model evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

models = {
    'Logistic Regression': log_reg,
    'Random Forest Classifier': rf_clf,
    'Support Vector Machine': svm
}

for model_name, model in models.items():
  print(f"Model: {model_name}")
  y_pred = model.predict(X_test)
  print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
  print(f"Classification Report:\n{classification_report(y_test, y_pred)}")
  print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
  print('-'*80)

Model: Logistic Regression
Accuracy: 0.6666666666666666
Classification Report:
              precision    recall  f1-score   support

          -1       0.50      1.00      0.67         1
           0       1.00      0.50      0.67         2

    accuracy                           0.67         3
   macro avg       0.75      0.75      0.67         3
weighted avg       0.83      0.67      0.67         3

Confusion Matrix:
[[1 0]
 [1 1]]
--------------------------------------------------------------------------------
Model: Random Forest Classifier
Accuracy: 1.0
Classification Report:
              precision    recall  f1-score   support

          -1       1.00      1.00      1.00         1
           0       1.00      1.00      1.00         2

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3

Confusion Matrix:
[[1 0]
 [0 2]]
---------------------------------------------------